<a href="https://colab.research.google.com/github/FadouaBOUAFIF/Cours_Computer_Vision_TP/blob/main/TP2_Segmentation_Otsu_KMeans_SIAA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 : Segmentation d'Images (Seuillage d'Otsu & K-Means)


## Objectifs
À l'issue de cette séance de travaux pratiques, l'étudiant sera capable de :
1. **Formuler et implémenter** par maximisation de la variance inter-classe $\sigma_B^2(t)$ (ou minimisation de la variance intra-classe $\sigma_W^2(t)$) et valider son seuil optimal face à OpenCV.
2. **Diagnostiquer les limites du seuillage global** face à un éclairage spatialement non uniforme ou un histogramme unimodal, et concevoir une variante adaptative locale par blocs.
3. **Implémenter l'algorithme de clustering K-Means** dans des espaces de caractéristiques multidimensionnels (intensité, espace couleur perceptuel CIELab, descripteurs spatiaux et de texture locale).
4. **Évaluer quantitativement** une chaîne complète d'inspection industrielle automatisée (segmentation + analyse en composantes connexes) à l'aide des métriques **Précision, Rappel, F1-score et IoU (Indice de Jaccard)**.

## Étape 0 : Préparation de l'environnement et chargement des données
Iimporter les bibliothèques scientifiques (`numpy`, `opencv-python`, `matplotlib`, `scikit-image`) et charger les images de référence.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['image.cmap'] = 'gray'

coins = data.coins()
page = data.page()
astronaut = data.astronaut()
print(f"Images chargées — coins: {coins.shape}, page: {page.shape}, astronaut: {astronaut.shape}")

---
## Étape 1 : Binarisation automatique par la méthode d'Otsu

### 1.1 Implémentation *From Scratch* de l'algorithme d'Otsu
**Rappel théorique :** Soit $p_i = \frac{h(i)}{N}$ la probabilité normalisée du niveau de gris $i \in [0, L-1]$. Pour tout seuil candidat $t \in \{0, \dots, L-2\}$ séparant les pixels en deux classes $C_0 = [0, t]$ (fond) et $C_1 = [t+1, L-1]$ (objet) :
- **Probabilités des classes :**
  $$\omega_0(t) = \sum_{i=0}^{t} p_i, \qquad \omega_1(t) = 1 - \omega_0(t)$$
- **Moyennes des classes :**
  $$\mu_0(t) = \frac{\sum_{i=0}^{t} i \cdot p_i}{\omega_0(t)}, \qquad \mu_1(t) = \frac{\sum_{i=t+1}^{L-1} i \cdot p_i}{\omega_1(t)}$$
- **Variance inter-classe (*Between-Class Variance*) :**
  $$\sigma_B^2(t) = \omega_0(t)\,\omega_1(t)\,[\mu_0(t) - \mu_1(t)]^2$$
- **Seuil optimal :** $T_{\text{opt}} = \arg\max_{t} \sigma_B^2(t)$.

**Consignes :**
1. Compléter la fonction `otsu_threshold_scratch(img)` qui retourne `(T_opt, mask_bin, variances_inter)`.
2. Valider votre implémentation sur une matrice numérique $6 \times 6$ puis sur `data.coins()` en comparant $T_{\text{opt}}$ avec `cv2.threshold(coins, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)`.
3. Tracer sur une même figure : (a) l'image originale, (b) l'histogramme avec la position de $T_{\text{opt}}$ et la courbe $\sigma_B^2(t)$ normalisée, (c) le masque binaire résultant.

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 1.1
# ======================================================================

def otsu_threshold_scratch(img, L=256):
    """
    Calcule le seuil optimal d'Otsu par maximisation de la variance inter-classe.
    Retourne : T_opt (int), mask_bin (uint8, valeurs 0 et 255), variances (ndarray de taille L)
    """
    variances = np.zeros(L, dtype=np.float64)
    # VOTRE CODE ICI (gérer les cas où omega_0 == 0 ou omega_1 == 0)
    T_opt = 0
    mask_bin = None
    return T_opt, mask_bin, variances

# --- Validation numérique sur matrice 6x6 bimodale ---
mat_bimodale = np.array([
    [10, 12, 11, 14, 10, 13],
    [12, 15, 11, 13, 12, 11],
    [11, 13, 210, 220, 14, 12],
    [10, 14, 215, 225, 11, 10],
    [13, 11, 12, 10, 14, 12],
    [11, 12, 13, 11, 12, 10]
], dtype=np.uint8)

# VOTRE CODE ICI : Tester sur mat_bimodale, puis sur data.coins() et comparer à cv2.THRESH_OTSU

### 1.2 Limites d'Otsu global face au gradient d'illumination et approche locale par blocs
**Contexte industriel :** Sur une ligne de production, un éclairage rasant ou décentré induit un gradient de luminance continu à travers le champ de vision (illustré par l'image `data.page()`).

**Consignes :**
1. Appliquer votre fonction `otsu_threshold_scratch` sur `data.page()` et analyser l'échec du seuillage global.
2. Implémenter `otsu_local_blocs(img, grille=(4, 4))` qui subdivise l'image en sous-blocs et calcule un seuil d'Otsu local par tuile.
3. Comparer visuellement : (a) Otsu global, (b) Otsu local par blocs ($4\times 4$ et $8\times 8$), et (c) le seuillage adaptatif gaussien (`cv2.adaptiveThreshold`).

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 1.2
# ======================================================================

def otsu_local_blocs(img, grille=(4, 4)):
    """
    Applique la binarisation d'Otsu indépendamment sur chaque sous-bloc de la grille.
    """
    h, w = img.shape
    out = np.zeros_like(img, dtype=np.uint8)
    # VOTRE CODE ICI
    return out

# VOTRE CODE ICI : Comparer Otsu global, otsu_local_blocs(page, (4,4)), (8,8) et cv2.adaptiveThreshold

---
## Étape 2 : Segmentation par clustering non supervisé K-Means

### 2.1 Implémentation *From Scratch* de l'algorithme K-Means
**Rappel théorique :** Étant donné un ensemble de $N$ vecteurs de caractéristiques $\{x_1, \dots, x_N\} \subset \mathbb{R}^d$ et un nombre de clusters $K$, l'algorithme K-Means minimise l'inertie intra-classe (*Within-Cluster Sum of Squares*, WCSS) :
$$D = \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - m_k \|_2^2$$
par alternance entre :
1. **Étape d'affectation :** $c_i = \arg\min_{k \in \{1,\dots,K\}} \| x_i - m_k \|_2^2$
2. **Étape de mise à jour :** $m_k = \frac{1}{|C_k|} \sum_{x_i \in C_k} x_i$
jusqu'à convergence ($\max_k \|m_k^{(t+1)} - m_k^{(t)}\| < \epsilon$ ou atteinte de `max_iter`).

**Consignes :**
1. Implémenter `kmeans_scratch(X, K, max_iter=100, tol=1e-4, seed=42)` de manière vectorisée (NumPy).
2. Segmenter l'image couleur `astronaut` (redimensionnée en $128 \times 128$ pour la rapidité) dans l'espace **RGB** puis dans l'espace perceptuel **CIELab** (`cv2.cvtColor(..., cv2.COLOR_RGB2LAB)`) pour $K \in \{2, 3, 4, 6\}$.
3. Tracer la courbe d'inertie $D(K)$ (méthode du coude / *Elbow Method*) et la courbe de convergence $D(\text{itération})$.

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 2.1
# ======================================================================

def kmeans_scratch(X, K, max_iter=100, tol=1e-4, seed=42):
    """
    Algorithme K-Means vectorisé sur une matrice X de dimension (N, d).
    Retourne : labels (N,), centroids (K, d), historique_D (liste des inerties par itération)
    """
    rng = np.random.default_rng(seed)
    N, d = X.shape
    # Initialisation aléatoire de K centroïdes distincts parmi les échantillons
    indices_init = rng.choice(N, size=K, replace=False)
    centroids = X[indices_init].astype(np.float64).copy()
    historique_D = []
    labels = np.zeros(N, dtype=np.int32)

    # VOTRE CODE ICI : Boucle d'affectation, calcul de D, mise à jour (gérer le cas cluster vide)

    return labels, centroids, historique_D

# VOTRE CODE ICI : Tester sur astronaut (128x128) en RGB et CIELab, puis tracer D(K) pour K=2..6

---
## Étape 3 : Mini-Projet — Inspection automatisée de circuits imprimés (PCB)

Dans une chaîne de contrôle qualité électronique, on souhaite détecter et classifier automatiquement deux types de défauts critiques sur des pistes de cuivre d'un circuit imprimé (PCB) en présence d'un éclairage non uniforme et de bruit capteur :
- **Coupure de piste** (*open circuit / track break*)
- **Court-circuit / pont de soudure** (*short circuit / solder bridge*)

**Protocole expérimental :**
1. Utiliser le générateur `generer_pcb_industriel()` fourni ci-dessous qui produit une image dégradée `img_pcb` (avec gradient d'illumination + bruit gaussien) ainsi que le masque de vérité terrain des défauts `gt_defauts`.
2. Construire pour chaque pixel un **vecteur de caractéristiques enrichi** normalisé ($\mu=0, \sigma=1$) :
   $$x_i = \left[ I(r,c),\; \mu_{\text{local}, 7\times 7}(r,c),\; \sigma_{\text{local}, 7\times 7}(r,c),\; I(r,c) - \mu_{\text{local}, 25\times 25}(r,c) \right]$$
   *(Remarque : le 4e descripteur est un filtrage passe-haut local qui annule le gradient d'éclairage !)*
3. Segmenter l'image par `kmeans_scratch` ($K=3$ : substrat, pistes saines, anomalies/défauts) ou extraire les défauts par comparaison différentielle et analyse en composantes connexes (`cv2.connectedComponentsWithStats`).
4. Filtrer les composantes parasites (bruit isolé d'aire $< 15$ pixels) et calculer les métriques quantitatives de détection par rapport à `gt_defauts` :
   $$\text{Précision} = \frac{TP}{TP + FP}, \quad \text{Rappel} = \frac{TP}{TP + FN}, \quad \text{IoU} = \frac{TP}{TP + FP + FN}, \quad F_1 = \frac{2\,TP}{2\,TP + FP + FN}$$

In [ ]:
# ======================================================================
# ESPACE D'IMPLÉMENTATION ÉTUDIANT — ÉTAPE 3
# ======================================================================

def generer_pcb_industriel(h=300, w=420, seed=42):
    rng = np.random.default_rng(seed)
    pcb_ref = np.ones((h, w), dtype=np.float64) * 45.0
    gt_defauts = np.zeros((h, w), dtype=np.uint8)

    # 3 pistes horizontales de cuivre + pastilles
    for y in [65, 145, 225]:
        cv2.rectangle(pcb_ref, (25, y), (395, y + 22), 195.0, -1)
        cv2.circle(pcb_ref, (25, y + 11), 16, 210.0, -1)
        cv2.circle(pcb_ref, (395, y + 11), 16, 210.0, -1)

    pcb_test = pcb_ref.copy()
    # Défaut 1 : Coupure de piste sur la piste supérieure
    cv2.rectangle(pcb_test, (175, 65), (195, 87), 45.0, -1)
    cv2.rectangle(gt_defauts, (175, 65), (195, 87), 255, -1)

    # Défaut 2 : Court-circuit (pont de soudure) entre piste 2 et piste 3
    cv2.rectangle(pcb_test, (275, 167), (292, 225), 195.0, -1)
    cv2.rectangle(gt_defauts, (275, 167), (292, 225), 255, -1)

    # Ajout d'un gradient d'illumination spatial + bruit capteur gaussien
    X_grid, Y_grid = np.meshgrid(np.linspace(-35, 45, w), np.linspace(-20, 25, h))
    bruit = rng.normal(0, 8.0, (h, w))
    img_pcb = np.clip(pcb_test + X_grid + Y_grid + bruit, 0, 255).astype(np.uint8)
    img_ref_acq = np.clip(pcb_ref + X_grid + Y_grid + rng.normal(0, 8.0, (h, w)), 0, 255).astype(np.uint8)
    return img_pcb, img_ref_acq, gt_defauts

def evaluer_segmentation(mask_pred, mask_gt):
    # VOTRE CODE ICI : Calculer Précision, Rappel, F1-score et IoU
    pass

# VOTRE CODE ICI :
# 1. Comparer Otsu global directement sur img_pcb (observer l'échec dû au gradient)
# 2. Extraire les descripteurs locaux + segmenter par K-Means (ou différence normalisée + K-Means)
# 3. Filtrer par composantes connexes et afficher les boîtes englobantes + scores IoU / F1

---
## Question de synthèse:

> **Problématique scientifique :**  
> Dans le cadre de la conception d'un système d'inspection visuelle industrielle temps réel, vous devez justifier le choix et l'architecture de l'algorithme de segmentation en amont d'un module de décision qualité :
> 1. **Analyse mathématique et algorithmique :** Démontrez l'équivalence exacte entre la maximisation de la variance inter-classe $\sigma_B^2(t)$ et la minimisation de la variance intra-classe $\sigma_W^2(t) = \omega_0(t)\sigma_0^2(t) + \omega_1(t)\sigma_1^2(t)$ dans la méthode d'Otsu. Expliquez ensuite pourquoi K-Means avec $K=2$ sur l'intensité scalaire $I(r,c)$ converge vers un seuil très proche d'Otsu, tout en présentant une sensibilité différente à l'initialisation et aux classes déséquilibrées.
> 2. **Espaces de représentation et invariance :** Pourquoi le clustering K-Means appliqué directement dans l'espace couleur **RGB** produit-il fréquemment une sur-segmentation des ombres et reflets spéculaires sur des pièces métalliques ou plastiques, et en quoi le passage aux espaces **CIELab** ou **HSV** (combiné à des descripteurs de texture locale) résout-il partiellement ce problème ?
> 3. **Limites du non-supervisé et hybridation :** Face à des défauts industriels rares, polymorphes ou à très faible contraste (*zero-day defects*), quelles sont les limites fondamentales des méthodes statistiques/non supervisées (Otsu, K-Means) et comment les hybrider efficacement avec des approches d'apprentissage profond (ex. auto-encodeurs de reconstruction, segmentation sémantique U-Net ou modèles fondation SAM) sous contrainte de frugalité en données annotées ?

### Rédigez votre synthèse argumentée dans la cellule ci-dessous :

**Réponse de l'étudiant :**

1. *Équivalence Otsu ($\sigma_B^2$ vs $\sigma_W^2$) et lien avec K-Means ($K=2$) :*  
   *(À compléter...)*

2. *Impact de l'espace de caractéristiques (RGB vs CIELab/HSV + texture) :*  
   *(À compléter...)*

3. *Limites du clustering non supervisé et stratégies d'hybridation Deep Learning :*  
   *(À compléter...)*